# ML Signal Filter — SVM (L1) + Triple Barrier Labeling

**Base strategy** (long only): plain mean reversion on the lower Bollinger Band, with an ATR-based stop
loss and a symmetric take profit (RR = 1). Entry is `close < BB lower`, optionally gated by an
EMA(100) trend filter (`close > EMA`) toggled with `USE_EMA_TREND_FILTER`.

**ML filter**: triple-barrier labels on the strategy's own TP/SL train a linear, L1-penalized SVM,
fit on a rolling training window, that decides whether to take or skip each signal (meta-labeling).

**Validation**: Each fold optimizes strategy parameters on a recent
window, trains the SVM on the following window, then tests both the base strategy and the
SVM-filtered version out-of-sample.

## Notebook structure
1. Configuration
2. Load data
3. Indicators & labeling
4. Features
5. Dataset-building pipeline
6. Strategy parameter grid search
7. Single fold walkthrough
8. Walk-forward across all folds
9. Walk-forward evaluation
10. Pooled trade-level significance tests

## 1. Configuration

In [428]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

pio.templates.default = "plotly_dark"


In [429]:
# ── Data ────────────────────────────────────────────────────────────────
CSV_PATH = "nasdaq_h1.csv"   # path to the Nasdaq OHLCV CSV — pick the file matching BAR_MINUTES below
                               # Kaggle export: DateTime, Open, High, Low, Close, Volume, TickVolume;
                               # tab-separated, dotted timestamps, descending time order

# ── Bar-resolution scaling ──────────────────────────────────────────────
# BAR_MINUTES is the actual candle resolution of CSV_PATH (15 for M15, 60 for H1, 240 for H4, ...).
# BAR_SCALE rescales every bar-counted parameter below so it keeps the same real-time meaning it
# would have on 1H data; bars() rounds the scaled value to a valid (>= 1) integer bar count, since a
# coarser timeframe like H4 doesn't divide the reference periods evenly.
BAR_MINUTES = 60
BAR_SCALE = 60 / BAR_MINUTES


def bars(n: float) -> int:
    return max(1, round(n * BAR_SCALE))


# ── Base strategy parameters (fixed, not gridded) ──────────────────────
HOURS_PER_DAY = 23
ATR_PERIOD  = bars(HOURS_PER_DAY)
K_SL        = 1.0     # stop distance = K_SL * ATR
RR          = 1.0     # symmetric take profit (reward:risk = 1)
MAX_HORIZON = bars(HOURS_PER_DAY * 5)   # vertical barrier, in bars
USE_EMA_TREND_FILTER = False   # if True, only take entries where close > EMA(EMA_FEATURE_PERIOD)

# ── Feature parameters ──────────────────────────────────────────────────
ADX_PERIOD           = bars(HOURS_PER_DAY)
RSI_PERIOD           = bars(HOURS_PER_DAY)
EMA_FEATURE_PERIOD   = bars(HOURS_PER_DAY * 5)   # fixed EMA trend filter above
PCT_ABOVE_EMA_WINDOW = bars(HOURS_PER_DAY * 5)
VOLUME_RATIO_WINDOW  = bars(HOURS_PER_DAY * 5)
SWING_LOOKBACK       = bars(10)   # bars each side used to confirm a swing low
BARS_PER_DAY         = bars(HOURS_PER_DAY)   # 1 trading day, in bars
SLOPE_WINDOW         = bars(HOURS_PER_DAY * 5)   # price-slope feature lookback (5 trading days)

# ── Strategy parameter grid (coarse) ────────────────────────────────────
# Strategy is plain Bollinger Band mean reversion, optionally gated by the EMA trend filter above —
# neither the trend filter nor its period is gridded, so only BB itself is.
BB_PERIOD_GRID = [10, 15, 20, 25, 30, 35, 40]
BB_STD_GRID    = [1.0, 1.5, 2.0, 2.5, 3.0]

# ── Fold / walk-forward windows (months, so 6-month windows are expressible) ──
FOLD_START       = "2017-01-01"
STRAT_OPT_MONTHS = 24   # strategy-parameter optimization window
ML_OPT_MONTHS    = 12    # ML training window
TEST_MONTHS      = 6    # out-of-sample test window
STEP_MONTHS      = TEST_MONTHS   # roll forward by one test window each fold

# ── Minimum trade counts, derived from a per-year rate ──────────────────
MIN_TRADES_PER_YEAR = int(50 * BAR_SCALE)
MIN_TRADES_OPT = round(MIN_TRADES_PER_YEAR * STRAT_OPT_MONTHS / 12)
MIN_TRADES_ML  = round(MIN_TRADES_PER_YEAR * ML_OPT_MONTHS / 12)

# ── SVM: linear, L1-penalized, fixed regularization ─────────────────────
SVM_C = 1.0
RANDOM_STATE = 42

## 2. Load data

`Volume` is 0 throughout this export — `TickVolume` is used instead.

In [430]:
df_raw = pd.read_csv(CSV_PATH, sep="\t")
df_raw["DateTime"] = pd.to_datetime(df_raw["DateTime"], format="%Y.%m.%d %H:%M:%S")
df_raw = df_raw.set_index("DateTime").sort_index()
df_raw.index.name = "time"
df_raw.columns = [c.lower() for c in df_raw.columns]

if "volume" in df_raw.columns and (df_raw["volume"] > 0).any():
    pass
elif "tickvolume" in df_raw.columns:
    df_raw["volume"] = df_raw["tickvolume"]
else:
    df_raw["volume"] = np.nan

df_raw = df_raw[["open", "high", "low", "close", "volume"]]
print(f"{len(df_raw)} bars  |  {df_raw.index.min()} -> {df_raw.index.max()}")
df_raw.head()


51889 bars  |  2016-11-15 06:00:00 -> 2025-10-01 07:00:00


,open,high,low,close,volume
time,,,,,
2016-11-15 06:00:00,4699.6,4703.1,4699.6,4701.6,449
2016-11-15 07:00:00,4701.7,4703.1,4698.7,4701.8,1387
2016-11-15 08:00:00,4701.7,4702.4,4697.8,4699.1,1068
2016-11-15 09:00:00,4699.0,4703.9,4697.6,4702.1,2158
2016-11-15 10:00:00,4702.2,4715.7,4699.7,4711.7,6117


In [453]:
fig = go.Figure(go.Scatter(x=df_raw.index, y=df_raw["close"], mode="lines", line=dict(width=2)))
fig.update_layout(title="Close price — full history", xaxis_title="Date", yaxis_title="Price", height=700)
fig.show()


## 3. Indicators & labeling

EMA and ATR use Wilder-style smoothing (`alpha = 1/period`).

In [432]:
def get_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()


def get_bollinger_bands(series: pd.Series, period: int, num_std: float):
    mid = series.rolling(period).mean()
    std = series.rolling(period).std()
    return mid + num_std * std, mid, mid - num_std * std


def get_true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    return pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)


def get_atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int, smoothing: bool = True) -> pd.Series:
    tr = get_true_range(high, low, close)
    return tr.ewm(alpha=1 / period, adjust=False).mean() if smoothing else tr.rolling(period).mean()


def get_rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def get_adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=high.index)

    atr = get_atr(high, low, close, period, smoothing=True)
    plus_di = 100 * plus_dm.ewm(alpha=1 / period, adjust=False).mean() / atr
    minus_di = 100 * minus_dm.ewm(alpha=1 / period, adjust=False).mean() / atr

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    return dx.ewm(alpha=1 / period, adjust=False).mean()


In [ ]:
def get_triple_barrier_labels(
    df: pd.DataFrame,
    signal_col="entry_long", 
    price_col="close", 
    open_col="open", 
    high_col="high", 
    low_col="low",
    atr_period=15, 
    k_sl=1.0, 
    rr=1.0, 
    max_horizon=20,
    precomputed_atr=None, 
    precomputed_signal=None,
) -> pd.DataFrame:
    """Upper barrier = TP, lower barrier = SL, vertical barrier = max_horizon bars.
    Entry price is the next bar's open (signal at close, execution at next open)."""
    atr = precomputed_atr if precomputed_atr is not None else get_atr(
        df[high_col], df[low_col], df[price_col], atr_period, smoothing=True
    )
    signal = (precomputed_signal if precomputed_signal is not None else df[signal_col]).values

    entry_prices = df[open_col].shift(-1).values
    risk = (atr * k_sl).values
    high, low, close = df[high_col].values, df[low_col].values, df[price_col].values
    n = len(df)

    label = np.full(n, np.nan)
    r_multiple = np.full(n, np.nan)
    bars_to_exit = np.full(n, np.nan)

    for i in np.where(signal != 0)[0]:
        if i >= n - 1:
            continue

        direction = np.sign(signal[i])
        r = risk[i]
        if r <= 0 or np.isnan(r):
            continue

        entry_price = entry_prices[i]
        sl_price = entry_price - direction * r
        tp_price = entry_price + direction * r * rr

        end = min(i + max_horizon, n - 1)
        window_high = high[i + 1: end + 1]
        window_low = low[i + 1: end + 1]

        if direction > 0:
            tp_hit = np.where(window_high >= tp_price)[0]
            sl_hit = np.where(window_low <= sl_price)[0]
        else:
            tp_hit = np.where(window_low <= tp_price)[0]
            sl_hit = np.where(window_high >= sl_price)[0]

        first_tp = tp_hit[0] if len(tp_hit) > 0 else np.inf
        first_sl = sl_hit[0] if len(sl_hit) > 0 else np.inf

        if first_tp == np.inf and first_sl == np.inf:
            r_multiple[i] = direction * (close[end] - entry_price) / r
            label[i] = 0
            bars_to_exit[i] = end - i
        elif first_tp < first_sl:
            label[i], r_multiple[i], bars_to_exit[i] = 1, rr, first_tp + 1
        else:
            label[i], r_multiple[i], bars_to_exit[i] = -1, -1.0, first_sl + 1

    result = pd.DataFrame({"label": label, "r_multiple": r_multiple, "bars_to_exit": bars_to_exit}, index=df.index)
    return result.dropna(how="all")


Illustrative view — real indicators on one month of data. Bollinger Bands use mid-grid parameters purely for display (optimization happens in section 6); the EMA is the one used by the `pct_above_ema` feature, and also gates entries as a trend filter when `USE_EMA_TREND_FILTER` is on (see §5).

In [434]:
illustration_start, illustration_end = "2019-01-01", "2019-02-01"
mask = (df_raw.index >= illustration_start) & (df_raw.index < illustration_end)

demo_ema = get_ema(df_raw["close"], EMA_FEATURE_PERIOD)
demo_bb_upper, demo_bb_mid, demo_bb_lower = get_bollinger_bands(df_raw["close"], 80, 1.5)
demo_rsi = get_rsi(df_raw["close"], RSI_PERIOD)
demo_adx = get_adx(df_raw["high"], df_raw["low"], df_raw["close"], ADX_PERIOD)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.6, 0.2, 0.2], vertical_spacing=0.05,
                     subplot_titles=("Price, Bollinger Bands (strategy), EMA (feature only)", "RSI", "ADX"))

fig.add_trace(go.Scatter(x=df_raw.index[mask], y=df_raw["close"][mask], name="Close",
                          line=dict(color="white", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_raw.index[mask], y=demo_ema[mask], name=f"EMA({EMA_FEATURE_PERIOD}, feature)",
                          line=dict(width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_raw.index[mask], y=demo_bb_upper[mask], name="BB(80, 1.5)",
                          line=dict(color="gray", dash="dash", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_raw.index[mask], y=demo_bb_lower[mask], showlegend=False,
                          line=dict(color="gray", dash="dash", width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=df_raw.index[mask], y=demo_rsi[mask], line=dict(color="purple", width=2),
                          showlegend=False), row=2, col=1)
fig.add_hline(y=70, line=dict(color="gray", dash="dot", width=2), row=2, col=1)
fig.add_hline(y=30, line=dict(color="gray", dash="dot", width=2), row=2, col=1)

fig.add_trace(go.Scatter(x=df_raw.index[mask], y=demo_adx[mask], line=dict(color="orange", width=2),
                          showlegend=False), row=3, col=1)

fig.update_layout(height=700, title="Illustrative indicators, one month")
fig.show()


**Triple barrier labeling — illustrative example.** One real signal (`close < BB lower`, using the
same demo Bollinger Bands as above), walked forward bar by bar: whichever of the take-profit,
stop-loss or vertical (`MAX_HORIZON`-bar) barrier is touched first sets the label.

In [435]:
demo_atr = get_atr(df_raw["high"], df_raw["low"], df_raw["close"], ATR_PERIOD, smoothing=True)
demo_raw_signal = (df_raw["close"] < demo_bb_lower).astype(int)
demo_entry_long = ((demo_raw_signal == 1) & (demo_raw_signal.shift(1) != 1)).astype(int)

demo_signal_times = demo_entry_long[demo_entry_long == 1].index
in_window = demo_signal_times >= illustration_start
demo_signal_time = demo_signal_times[in_window][0] if in_window.any() else demo_signal_times[0]
i = df_raw.index.get_loc(demo_signal_time)

entry_time = df_raw.index[i + 1]
entry_price = df_raw["open"].iloc[i + 1]
risk = demo_atr.iloc[i] * K_SL
tp_price = entry_price + risk * RR
sl_price = entry_price - risk

end = min(i + MAX_HORIZON, len(df_raw) - 1)
path = df_raw.iloc[i + 1: end + 1]
tp_hit = path.index[path["high"] >= tp_price]
sl_hit = path.index[path["low"] <= sl_price]

if len(tp_hit) and (not len(sl_hit) or tp_hit[0] <= sl_hit[0]):
    exit_time, exit_price, outcome, exit_color = tp_hit[0], tp_price, "TP hit (label = 1)", "seagreen"
elif len(sl_hit):
    exit_time, exit_price, outcome, exit_color = sl_hit[0], sl_price, "SL hit (label = -1)", "firebrick"
else:
    exit_time = df_raw.index[end]
    exit_price, outcome, exit_color = df_raw["close"].iloc[end], "Vertical barrier (label = 0)", "gray"

plot_start = df_raw.index[max(i - 15, 0)]
plot_end = df_raw.index[min(end + 10, len(df_raw) - 1)]
plot_df = df_raw.loc[plot_start:plot_end]

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df["close"], mode="lines",
                          line=dict(color="white", width=2), name="Close"))
fig.add_trace(go.Scatter(x=[demo_signal_time], y=[df_raw["close"].loc[demo_signal_time]], mode="markers",
                          marker=dict(color="orange", size=14, symbol="triangle-up"),
                          name="Signal (close < BB lower)"))
fig.add_trace(go.Scatter(x=[entry_time], y=[entry_price], mode="markers",
                          marker=dict(color="white", size=12, symbol="circle"), name="Entry (next open)"))
fig.add_trace(go.Scatter(x=[exit_time], y=[exit_price], mode="markers",
                          marker=dict(color=exit_color, size=16, symbol="star"), name=f"Exit — {outcome}"))

fig.add_hline(y=tp_price, line=dict(color="seagreen", dash="dash", width=2), annotation_text="TP barrier")
fig.add_hline(y=sl_price, line=dict(color="firebrick", dash="dash", width=2), annotation_text="SL barrier")
fig.add_vline(x=df_raw.index[end], line=dict(color="gray", dash="dot", width=2), annotation_text="Vertical barrier")

fig.update_layout(height=800, xaxis_title="Date", yaxis_title="Price",
                   title=f"Triple barrier labeling — illustrative example ({outcome})")
fig.show()

## 4. Features

| Feature | What it captures |
|---|---|
| `close_ema_dist_atr` | relative position of `close` vs. that same EMA — signed distance in ATR units |
| `adx` | trend strength |
| `rsi` | momentum |
| `candle_body_ratio` | signed candle body / range |
| `lower_wick_ratio` | lower wick / range |
| `volume_ratio` | volume vs. the last `VOLUME_RATIO_WINDOW` bars |
| `hour` | time of day |
| `dow` | day of week |
| `swing_low_dist_atr` | distance to the nearest confirmed recent swing low, in ATR units |
| `price_slope_atr` | OLS slope of `close` over the last `SLOPE_WINDOW` (`BARS_PER_DAY * 5`) bars, in ATR units per bar |
| `consecutive_red_candles` | number of consecutive bearish (`close < open`) candles ending at, and including, the signal candle |

A swing low at bar `j` needs bars `j - SWING_LOOKBACK` to `j + SWING_LOOKBACK` to confirm, so it's only
*known* starting at bar `j + SWING_LOOKBACK`. `get_swing_low_features` shifts detection forward by that
amount before filling, so no future information leaks into the feature.

In [436]:
FEATURE_COLS = [
    "close_ema_dist_atr", 
    "adx", 
    "rsi",
    "candle_body_ratio", 
    "lower_wick_ratio",
    "close_diff_atr", 
    "low_diff_atr", 
    "volume_ratio",
    "hour", 
    "dow", 
    "swing_low_dist_atr", 
    "price_slope_atr", 
    "consecutive_red_candles",
]


def get_swing_low_features(df: pd.DataFrame, atr: pd.Series, swing_window: int) -> pd.Series:
    low = df["low"]
    window = 2 * swing_window + 1
    is_swing_low = low == low.rolling(window, center=True, min_periods=window).min()

    known_swing_price = low.where(is_swing_low).shift(swing_window)
    swing_low_price = known_swing_price.ffill()
    return pd.Series((df["close"].values - swing_low_price.values) / atr.values,
                      index=df.index, name="swing_low_dist_atr")


def get_slope(series: pd.Series, window: int) -> pd.Series:
    """OLS slope of `series` over each trailing `window`-bar block, via rolling sums (vectorized —
    a rolling().apply() with a Python callback is orders of magnitude slower at this data size)."""
    x = np.arange(window)
    x_mean = x.mean()
    x_ss = ((x - x_mean) ** 2).sum()

    position = pd.Series(np.arange(len(series)), index=series.index, dtype=float)
    sum_y = series.rolling(window).sum()
    sum_position_y = (position * series).rolling(window).sum()
    window_start = position - window + 1
    sum_local_index_y = sum_position_y - window_start * sum_y
    return (sum_local_index_y - x_mean * sum_y) / x_ss


def get_consecutive_red_candles(df: pd.DataFrame) -> pd.Series:
    is_red = (df["close"] < df["open"]).astype(int)
    streak_id = (is_red != is_red.shift()).cumsum()
    streak_len = is_red.groupby(streak_id).cumcount() + 1
    return (streak_len * is_red).rename("consecutive_red_candles")


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    ema_feature = get_ema(out["close"], EMA_FEATURE_PERIOD)
    out["close_ema_dist_atr"] = (out["close"] - ema_feature) / out["atr"]
    out["adx"] = get_adx(out["high"], out["low"], out["close"], period=ADX_PERIOD)
    out["rsi"] = get_rsi(out["close"], period=RSI_PERIOD)

    candle_range = (out["high"] - out["low"]).replace(0, np.nan)
    body_bottom = out[["open", "close"]].min(axis=1)
    out["candle_body_ratio"] = (out["close"] - out["open"]) / candle_range
    out["lower_wick_ratio"] = (body_bottom - out["low"]) / candle_range
    out["close_diff_atr"] = out["close"].diff() / df["atr"]
    out["low_diff_atr"] = out["low"].diff() / df["atr"]

    out["volume_ratio"] = out["volume"] / out["volume"].rolling(VOLUME_RATIO_WINDOW).mean()

    out["hour"] = out.index.hour
    out["dow"] = out.index.dayofweek

    out["swing_low_dist_atr"] = get_swing_low_features(out, out["atr"], swing_window=SWING_LOOKBACK)
    out["price_slope_atr"] = get_slope(out["close"], SLOPE_WINDOW) / out["atr"]
    out["consecutive_red_candles"] = get_consecutive_red_candles(out)
    return out

## 5. Dataset-building pipeline

`build_dataset` recomputes indicators, signal, labels and features for one parameter combination, over
the **full** history — so indicators are properly warmed up before any fold's window starts.
`signal_start`/`signal_end` restrict where signals can fire without touching the indicators, keeping the
labeling loop fast.

`select_window_signals` extracts one window's ML-ready rows, dropping any signal whose outcome window
would extend past the window's end — otherwise its label would depend on the *next* window.

In [437]:
def add_indicators(df: pd.DataFrame, bb_period: int, bb_std: float, atr_period: int) -> pd.DataFrame:
    out = df.copy()
    out["bb_upper"], out["bb_mid"], out["bb_lower"] = get_bollinger_bands(out["close"], bb_period, bb_std)
    out["atr"] = get_atr(out["high"], out["low"], out["close"], atr_period, smoothing=True)
    return out.dropna().copy()


def add_signal(df: pd.DataFrame, signal_start=None, signal_end=None) -> pd.DataFrame:
    out = df.copy()
    raw_signal = out["close"] < out["bb_lower"]

    if USE_EMA_TREND_FILTER:
        ema_trend = get_ema(out["close"], EMA_FEATURE_PERIOD)
        raw_signal &= out["close"] > ema_trend

    raw = raw_signal.astype(int)
    entry_long = ((raw == 1) & (raw.shift(1) != 1)).astype(int)

    if signal_start is not None or signal_end is not None:
        keep = pd.Series(True, index=out.index)
        if signal_start is not None:
            keep &= out.index >= pd.Timestamp(signal_start)
        if signal_end is not None:
            keep &= out.index < pd.Timestamp(signal_end)
        entry_long = entry_long.where(keep, 0)

    out["entry_long"] = entry_long
    return out


def add_labels(df: pd.DataFrame, atr_period: int, k_sl: float, rr: float, max_horizon: int) -> pd.DataFrame:
    labels = get_triple_barrier_labels(
        df, signal_col="entry_long", atr_period=atr_period, k_sl=k_sl, rr=rr,
        max_horizon=max_horizon, precomputed_atr=df["atr"],
    )
    return df.join(labels)


def build_dataset(df_raw, bb_period, bb_std, atr_period, k_sl, rr, max_horizon,
                   signal_start=None, signal_end=None) -> pd.DataFrame:
    df = add_indicators(df_raw, bb_period, bb_std, atr_period)
    df = add_signal(df, signal_start=signal_start, signal_end=signal_end)
    df = add_labels(df, atr_period, k_sl, rr, max_horizon)
    return add_features(df)


def select_window_signals(df: pd.DataFrame, start, end, max_horizon: int) -> pd.DataFrame:
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    end_pos = df.index.searchsorted(end)
    pos = np.arange(len(df))

    in_window = (df.index >= start) & (df.index < end)
    resolved = (pos + max_horizon) < end_pos

    sig = df.loc[in_window & resolved & (df["entry_long"] == 1) & df["label"].notna()].copy()
    sig = sig.dropna(subset=FEATURE_COLS)
    sig["meta_label"] = (sig["label"] == 1).astype(int)
    return sig


## 6. Strategy parameter grid search

Coarse grid over `BB_PERIOD × BB_STD`, maximizing precision on the strategy-optimization window.
Combinations under `MIN_TRADES_OPT` trades are discarded before ranking — otherwise the search
gravitates toward parameter sets with a handful of lucky trades at inflated precision.

In [438]:
def optimize_strategy_params(df_raw, opt_start, opt_end, bb_period_grid, bb_std_grid,
                              atr_period, k_sl, rr, max_horizon, min_trades):
    rows = []
    for bb_period in bb_period_grid:
        for bb_std in bb_std_grid:
            df = build_dataset(df_raw, bb_period, bb_std, atr_period, k_sl, rr, max_horizon,
                                signal_start=opt_start, signal_end=opt_end)
            sig = select_window_signals(df, opt_start, opt_end, max_horizon)
            n_trades = len(sig)
            rows.append({
                "bb_period": bb_period, "bb_std": bb_std,
                "n_trades": n_trades,
                "precision": sig["meta_label"].mean() if n_trades > 0 else np.nan,
            })

    grid_results = pd.DataFrame(rows)
    valid = grid_results[grid_results["n_trades"] >= min_trades]
    if valid.empty:
        raise ValueError(
            f"No parameter combination reached {min_trades} trades in [{opt_start}, {opt_end}) "
            f"— lower MIN_TRADES_OPT or widen the grid."
        )

    best = valid.sort_values("precision", ascending=False).iloc[0]
    return best, grid_results


## 7. Single fold walkthrough (2017–2019)

`run_fold` covers one 24-month block: optimize strategy parameters (12mo), fit an SVM on the
following ML window (6mo, rolling — most recent window only), then compare it against the base
strategy on the same test window (6mo). `MIN_TRADES_OPT`/`MIN_TRADES_ML` guard each step; a fold
that falls short skips the ML model but keeps its base-strategy numbers.

In [439]:
def fit_svm(X: np.ndarray, y: np.ndarray) -> LinearSVC:
    return LinearSVC(penalty="l1", loss="squared_hinge", dual=False,
                      C=SVM_C, class_weight="balanced", random_state=RANDOM_STATE).fit(X, y)


def fit_and_evaluate(train_signals: pd.DataFrame, test_signals: pd.DataFrame, min_trades_ml: int,
                      fold_start, label: str) -> dict:
    empty = {f"{label}_trades": np.nan, f"{label}_wins": np.nan, f"{label}_precision": np.nan,
             f"{label}_n_nonzero_coefs": np.nan, f"{label}_kept_features": None}

    if len(train_signals) < min_trades_ml:
        print(f"Fold at {fold_start} [{label}]: {len(train_signals)} ML trades, "
              f"below MIN_TRADES_ML={min_trades_ml} — skipped.")
        return empty
    if len(test_signals) == 0 or train_signals["meta_label"].nunique() < 2:
        print(f"Fold at {fold_start} [{label}]: empty test window or single-class training labels — skipped.")
        return empty

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_signals[FEATURE_COLS])
    X_test = scaler.transform(test_signals[FEATURE_COLS])
    svm = fit_svm(X_train, train_signals["meta_label"].values)

    mask = svm.predict(X_test) == 1
    nonzero = np.abs(svm.coef_[0]) > 1e-10
    kept = [f for f, keep in zip(FEATURE_COLS, nonzero) if keep]
    wins = int(test_signals.loc[mask, "meta_label"].sum()) if mask.sum() > 0 else 0

    return {
        f"{label}_trades": int(mask.sum()), f"{label}_wins": wins,
        f"{label}_precision": wins / mask.sum() if mask.sum() > 0 else np.nan,
        f"{label}_n_nonzero_coefs": int(nonzero.sum()),
        f"{label}_kept_features": ", ".join(kept) if kept else "(none)",
    }


def run_fold(df_raw, fold_start, strat_opt_months, ml_opt_months, test_months,
             bb_period_grid, bb_std_grid, atr_period, k_sl, rr, max_horizon,
             min_trades_opt, min_trades_ml):

    strat_opt_start = pd.Timestamp(fold_start)
    strat_opt_end = strat_opt_start + pd.DateOffset(months=strat_opt_months)
    ml_opt_end = strat_opt_end + pd.DateOffset(months=ml_opt_months)
    test_start = ml_opt_end
    test_end = test_start + pd.DateOffset(months=test_months)

    ml_opt_start_rolling = strat_opt_end

    best_params, grid_results = optimize_strategy_params(
        df_raw, strat_opt_start, strat_opt_end, bb_period_grid, bb_std_grid,
        atr_period, k_sl, rr, max_horizon, min_trades_opt,
    )

    df = build_dataset(
        df_raw, int(best_params["bb_period"]), float(best_params["bb_std"]),
        atr_period, k_sl, rr, max_horizon,
        signal_start=ml_opt_start_rolling, signal_end=test_end,
    )

    train_signals_rolling = select_window_signals(df, ml_opt_start_rolling, ml_opt_end, max_horizon)
    test_signals = select_window_signals(df, test_start, test_end, max_horizon)

    result = {
        "strat_opt_start": strat_opt_start, "strat_opt_end": strat_opt_end,
        "ml_opt_start_rolling": ml_opt_start_rolling,
        "ml_opt_end": ml_opt_end, "test_start": test_start, "test_end": test_end,
        "bb_period": int(best_params["bb_period"]),
        "bb_std": float(best_params["bb_std"]),
        "opt_precision": best_params["precision"], "opt_trades": int(best_params["n_trades"]),
        "train_trades_rolling": len(train_signals_rolling),
        "test_trades": len(test_signals),
        "test_wins": int(test_signals["meta_label"].sum()) if len(test_signals) > 0 else np.nan,
        "base_precision": test_signals["meta_label"].mean() if len(test_signals) > 0 else np.nan,
    }

    result.update(fit_and_evaluate(train_signals_rolling, test_signals, min_trades_ml, fold_start, "rolling"))

    return result, grid_results, train_signals_rolling, test_signals

In [440]:
fold_result, grid_results, train_signals_rolling, test_signals = run_fold(
    df_raw, FOLD_START, STRAT_OPT_MONTHS, ML_OPT_MONTHS, TEST_MONTHS,
    BB_PERIOD_GRID, BB_STD_GRID, ATR_PERIOD, K_SL, RR, MAX_HORIZON,
    MIN_TRADES_OPT, MIN_TRADES_ML,
)

print(f"Best params: BB={fold_result['bb_period']}/{fold_result['bb_std']}")
print(f"Opt window:  precision={fold_result['opt_precision']:.3f}  ({fold_result['opt_trades']} trades)")
print(f"Test window: base precision={fold_result['base_precision']:.3f} ({fold_result['test_trades']} trades)")
print(f"  rolling   ({fold_result['train_trades_rolling']:4d} train): "
      f"precision={fold_result['rolling_precision']:.3f} ({fold_result['rolling_trades']} trades)  "
      f"|  L1 kept {fold_result['rolling_n_nonzero_coefs']}/{len(FEATURE_COLS)}: {fold_result['rolling_kept_features']}")

grid_results.sort_values("precision", ascending=False).head(10)

Best params: BB=15/1.0
Opt window:  precision=0.513  (679 trades)
Test window: base precision=0.449 (158 trades)
  rolling   ( 341 train): precision=0.410 (100 trades)  |  L1 kept 12/13: close_ema_dist_atr, adx, rsi, candle_body_ratio, lower_wick_ratio, low_diff_atr, volume_ratio, hour, dow, swing_low_dist_atr, price_slope_atr, consecutive_red_candles


,bb_period,bb_std,n_trades,precision
5,15,1.0,679,0.512518
6,15,1.5,563,0.493783
20,30,1.0,426,0.490610
0,10,1.0,867,0.490196
10,20,1.0,570,0.487719
1,10,1.5,646,0.487616
15,25,1.0,487,0.486653
8,15,2.5,188,0.484043
2,10,2.0,361,0.481994
25,35,1.0,386,0.481865


**Grid search results.** Left: precision by BB period × BB_STD. Middle: trade count for the same grid. Right: precision vs. trade count for all combinations — the danger zone (top-left) is exactly what `MIN_TRADES_OPT` rules out.

In [441]:
pivot_precision = grid_results.pivot_table(index="bb_period", columns="bb_std", values="precision")
pivot_trades = grid_results.pivot_table(index="bb_period", columns="bb_std", values="n_trades")

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Precision by BB period x BB_STD", "Trades by BB period x BB_STD", "Precision vs. sample size",
))

fig.add_trace(go.Heatmap(
    z=pivot_precision.values, x=[str(c) for c in pivot_precision.columns], y=[str(i) for i in pivot_precision.index],
    colorscale="Viridis", text=np.round(pivot_precision.values, 2), texttemplate="%{text}",
    colorbar=dict(x=0.28, len=0.9),
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=pivot_trades.values, x=[str(c) for c in pivot_trades.columns], y=[str(i) for i in pivot_trades.index],
    colorscale="Blues", text=pivot_trades.values, texttemplate="%{text}",
    colorbar=dict(x=0.635, len=0.9),
), row=1, col=2)

passed = grid_results["n_trades"] >= MIN_TRADES_OPT
fig.add_trace(go.Scatter(x=grid_results.loc[~passed, "n_trades"], y=grid_results.loc[~passed, "precision"],
                          mode="markers", marker=dict(color="lightgray"),
                          name=f"< {MIN_TRADES_OPT} trades"), row=1, col=3)
fig.add_trace(go.Scatter(x=grid_results.loc[passed, "n_trades"], y=grid_results.loc[passed, "precision"],
                          mode="markers", marker=dict(color="steelblue"), name="passes filter"), row=1, col=3)
chosen = grid_results.loc[passed].sort_values("precision", ascending=False).iloc[0]
fig.add_trace(go.Scatter(x=[chosen["n_trades"]], y=[chosen["precision"]], mode="markers",
                          marker=dict(color="red", size=16, symbol="star"), name="Chosen"), row=1, col=3)
fig.add_vline(x=MIN_TRADES_OPT, line=dict(color="gray", dash="dash", width=2), row=1, col=3)

fig.update_xaxes(title_text="BB_STD", row=1, col=1)
fig.update_yaxes(title_text="BB period", row=1, col=1)
fig.update_xaxes(title_text="BB_STD", row=1, col=2)
fig.update_yaxes(title_text="BB period", row=1, col=2)
fig.update_xaxes(title_text="Trades in opt window", row=1, col=3)
fig.update_yaxes(title_text="Precision", row=1, col=3)
fig.update_layout(height=600, width=1800, title="Strategy parameter grid search — fold 1")
fig.show()


**Fold timeline.** Strategy-opt, ML-training and test windows on the actual price series, with every entry signal marked.

In [442]:
demo_df = build_dataset(
    df_raw, fold_result["bb_period"], fold_result["bb_std"],
    ATR_PERIOD, K_SL, RR, MAX_HORIZON,
    signal_start=fold_result["strat_opt_start"], signal_end=fold_result["test_end"],
)
demo_df = demo_df.loc[fold_result["strat_opt_start"]:fold_result["test_end"]]
entries = demo_df[demo_df["entry_long"] == 1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=demo_df.index, y=demo_df["close"], mode="lines",
                          line=dict(color="white", width=2), name="Close"))
fig.add_trace(go.Scatter(x=entries.index, y=entries["close"], mode="markers",
                          marker=dict(color="red", size=20, symbol="triangle-up"), name="Entry signal"))

fig.add_vrect(x0=fold_result["strat_opt_start"], x1=fold_result["strat_opt_end"],
              fillcolor="blue", opacity=0.08, line_width=0, annotation_text="Strategy opt")
fig.add_vrect(x0=fold_result["ml_opt_start_rolling"], x1=fold_result["ml_opt_end"],
              fillcolor="orange", opacity=0.12, line_width=0, annotation_text="ML window")
fig.add_vrect(x0=fold_result["test_start"], x1=fold_result["test_end"],
              fillcolor="green", opacity=0.12, line_width=0, annotation_text="Test")

fig.update_layout(height=600, xaxis_title="Date", yaxis_title="Price",
                   title=f"Fold 1 — BB={fold_result['bb_period']}/{fold_result['bb_std']}")
fig.show()


**Feature distributions** for this fold's rolling-window ML training trades.

In [454]:
fig = make_subplots(rows=4, cols=4, subplot_titles=FEATURE_COLS)
for i, col in enumerate(FEATURE_COLS):
    r, c = divmod(i, 4)
    fig.add_trace(go.Histogram(x=train_signals_rolling[col].dropna(), marker_color="steelblue",
                                showlegend=False), row=r + 1, col=c + 1)
fig.update_layout(height=800, title="Feature distributions — fold 1 rolling ML window")
fig.show()

**What the rolling model actually did**: predicted-take vs. predicted-skip against the real outcome. The bottom-left cell is the opportunity cost of filtering — skipped signals that would have won.

In [444]:
scaler_demo = StandardScaler()
X_train_demo = scaler_demo.fit_transform(train_signals_rolling[FEATURE_COLS])
X_test_demo = scaler_demo.transform(test_signals[FEATURE_COLS])
svm_demo = fit_svm(X_train_demo, train_signals_rolling["meta_label"].values)
y_pred_demo = svm_demo.predict(X_test_demo)
y_true_demo = test_signals["meta_label"].values

cm = np.array([
    [((y_true_demo == 1) & (y_pred_demo == 1)).sum(), ((y_true_demo == 0) & (y_pred_demo == 1)).sum()],
    [((y_true_demo == 1) & (y_pred_demo == 0)).sum(), ((y_true_demo == 0) & (y_pred_demo == 0)).sum()],
])

fig = go.Figure(go.Heatmap(
    z=cm, x=["Actual win", "Actual loss"], y=["Predicted take", "Predicted skip"],
    colorscale="Blues", text=cm, texttemplate="%{text}", textfont=dict(size=16), showscale=False,
))
fig.update_layout(title="Rolling model — test window outcomes", height=800, width=800)
fig.show()


**SVM feature weights — fold 1, rolling model (first test split).** Standardized coefficients of
the L1-penalized SVM: a weight of exactly zero means the L1 penalty dropped that feature; the sign
shows whether it pushes the decision toward taking (positive) or skipping (negative) the signal.

In [445]:
coef_series = pd.Series(svm_demo.coef_[0], index=FEATURE_COLS).sort_values()

print("SVM feature weights — fold 1, rolling model (test split 1):")
for feat, w in coef_series.reindex(coef_series.abs().sort_values(ascending=False).index).items():
    print(f"  {feat:<25} {w:+.4f}")

fig = go.Figure(go.Bar(
    x=coef_series.values, y=coef_series.index, orientation="h",
    marker_color=["seagreen" if w > 0 else "firebrick" for w in coef_series.values],
))
fig.add_vline(x=0, line=dict(color="gray", width=2))
fig.update_layout(height=600, width=1700, title="SVM feature weights — fold 1, rolling model (test split 1)",
                   xaxis_title="Coefficient (standardized features)", yaxis_title="Feature")
fig.show()

SVM feature weights — fold 1, rolling model (test split 1):
  close_ema_dist_atr        +0.1416
  rsi                       -0.1010
  hour                      +0.0915
  price_slope_atr           -0.0723
  low_diff_atr              +0.0723
  volume_ratio              -0.0442
  consecutive_red_candles   +0.0386
  dow                       +0.0290
  lower_wick_ratio          +0.0242
  candle_body_ratio         -0.0200
  swing_low_dist_atr        -0.0073
  adx                       -0.0070
  close_diff_atr            +0.0000


**SVM decision boundary & margins — illustrative 2D projection.** The real rolling filter is fit on
all engineered features; a full decision boundary can't be drawn in that many dimensions. For
visualization only, a separate linear SVM with the same hyperparameters is refit on just the two
features with the largest \|coefficient\| in the fold-1 model above, so the boundary and its ±1
margins can be plotted directly against the training points.

In [446]:
coef_abs = np.abs(svm_demo.coef_[0])
top2 = np.argsort(coef_abs)[-2:][::-1]
feat_x, feat_y = FEATURE_COLS[top2[0]], FEATURE_COLS[top2[1]]

X_2d = X_train_demo[:, top2]
y_2d = train_signals_rolling["meta_label"].values
svm_2d = fit_svm(X_2d, y_2d)

pad = 1.0
x_min, x_max = X_2d[:, 0].min() - pad, X_2d[:, 0].max() + pad
y_min, y_max = X_2d[:, 1].min() - pad, X_2d[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
zz = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig = go.Figure()
fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=zz, showscale=False,
                          contours=dict(start=-1, end=1, size=2, coloring="lines"),
                          line=dict(width=2, dash="dash"), colorscale=[[0, "gray"], [1, "gray"]],
                          name="Margin (±1)"))
fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=zz, showscale=False,
                          contours=dict(start=0, end=0, size=1, coloring="lines"),
                          line=dict(width=3), colorscale=[[0, "white"], [1, "white"]],
                          name="Decision boundary"))
fig.add_trace(go.Scatter(x=X_2d[y_2d == 1, 0], y=X_2d[y_2d == 1, 1], mode="markers",
                          marker=dict(color="seagreen", size=9, symbol="circle"), name="Win"))
fig.add_trace(go.Scatter(x=X_2d[y_2d == 0, 0], y=X_2d[y_2d == 0, 1], mode="markers",
                          marker=dict(color="firebrick", size=9, symbol="x"), name="Loss"))

fig.update_layout(height=800, width=1600,
                   title=f"SVM decision boundary & margins — {feat_x} vs. {feat_y} (fold 1, rolling)",
                   xaxis_title=f"{feat_x} (standardized)", yaxis_title=f"{feat_y} (standardized)")
fig.show()

## 8. Walk-forward across all folds

Rolls the 24-month block forward by `STEP_MONTHS`, stopping once a fold's test window would run past
the end of the data.

In [447]:
def run_walk_forward(df_raw, first_fold_start, strat_opt_months, ml_opt_months, test_months, step_months,
                      bb_period_grid, bb_std_grid, atr_period, k_sl, rr, max_horizon,
                      min_trades_opt, min_trades_ml):
    data_end = df_raw.index.max()
    fold_start = pd.Timestamp(first_fold_start)
    fold_span_months = strat_opt_months + ml_opt_months + test_months

    results, fold_id = [], 0
    while fold_start + pd.DateOffset(months=fold_span_months) <= data_end:
        fold_id += 1
        res, _, _, _ = run_fold(
            df_raw, fold_start, strat_opt_months, ml_opt_months, test_months,
            bb_period_grid, bb_std_grid, atr_period, k_sl, rr, max_horizon,
            min_trades_opt, min_trades_ml,
        )
        res["fold_id"] = fold_id
        results.append(res)

        print(f"Fold {fold_id}: BB={res['bb_period']}/{res['bb_std']} "
              f"| opt precision={res['opt_precision']:.3f} ({res['opt_trades']} trades) "
              f"| test base={res['base_precision']:.3f} "
              f"rolling={res['rolling_precision']:.3f} ({res['train_trades_rolling']}\u2192{res['rolling_trades']})")

        fold_start += pd.DateOffset(months=step_months)

    return pd.DataFrame(results)

In [448]:
wf_results = run_walk_forward(
    df_raw, FOLD_START, STRAT_OPT_MONTHS, ML_OPT_MONTHS, TEST_MONTHS, STEP_MONTHS,
    BB_PERIOD_GRID, BB_STD_GRID, ATR_PERIOD, K_SL, RR, MAX_HORIZON,
    MIN_TRADES_OPT, MIN_TRADES_ML,
)

wf_results[[
    "fold_id", "test_start", "test_end", "bb_period", "bb_std",
    "opt_precision", "opt_trades", "test_trades", "base_precision",
    "train_trades_rolling", "rolling_trades", "rolling_precision",
]]

Fold 1: BB=15/1.0 | opt precision=0.513 (679 trades) | test base=0.449 rolling=0.410 (341→100)
Fold 2: BB=30/1.0 | opt precision=0.506 (453 trades) | test base=0.587 rolling=0.618 (213→55)
Fold 3: BB=10/1.0 | opt precision=0.524 (901 trades) | test base=0.440 rolling=0.482 (423→112)
Fold 4: BB=10/1.0 | opt precision=0.515 (888 trades) | test base=0.461 rolling=0.467 (451→122)
Fold 5: BB=35/1.0 | opt precision=0.549 (395 trades) | test base=0.523 rolling=0.590 (211→39)
Fold 6: BB=25/1.5 | opt precision=0.527 (395 trades) | test base=0.564 rolling=0.750 (225→44)
Fold 7: BB=35/1.0 | opt precision=0.545 (418 trades) | test base=0.485 rolling=0.480 (232→25)
Fold 8: BB=35/1.0 | opt precision=0.540 (433 trades) | test base=0.534 rolling=0.623 (223→53)
Fold 9: BB=35/1.0 | opt precision=0.516 (448 trades) | test base=0.482 rolling=0.459 (209→61)
Fold 10: BB=35/1.0 | opt precision=0.521 (447 trades) | test base=0.505 rolling=0.478 (220→46)
Fold 11: BB=30/1.5 | opt precision=0.533 (411 trades) | 

,fold_id,test_start,test_end,bb_period,bb_std,opt_precision,opt_trades,test_trades,base_precision,train_trades_rolling,rolling_trades,rolling_precision
0,1,2020-01-01,2020-07-01,15,1.0,0.512518,679,158,0.449367,341,100,0.410000
1,2,2020-07-01,2021-01-01,30,1.0,0.505519,453,109,0.587156,213,55,0.618182
2,3,2021-01-01,2021-07-01,10,1.0,0.523862,901,225,0.440000,423,112,0.482143
3,4,2021-07-01,2022-01-01,10,1.0,0.514640,888,243,0.460905,451,122,0.467213
4,5,2022-01-01,2022-07-01,35,1.0,0.549367,395,111,0.522523,211,39,0.589744
5,6,2022-07-01,2023-01-01,25,1.5,0.526582,395,110,0.563636,225,44,0.750000
6,7,2023-01-01,2023-07-01,35,1.0,0.545455,418,103,0.485437,232,25,0.480000
7,8,2023-07-01,2024-01-01,35,1.0,0.540416,433,103,0.533981,223,53,0.622642
8,9,2024-01-01,2024-07-01,35,1.0,0.515625,448,114,0.482456,209,61,0.459016
9,10,2024-07-01,2025-01-01,35,1.0,0.521253,447,93,0.505376,220,46,0.478261


## 9. Walk-forward evaluation

Precision and trade counts per fold, comparing the base strategy against the rolling ML filter — plus
how many signals the filter actually keeps versus the full unfiltered set.

In [449]:
def wilson_ci(wins, n, z=1.96):
    if n == 0 or pd.isna(n):
        return np.nan, np.nan
    phat = wins / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin


valid = wf_results.dropna(subset=["base_precision", "rolling_precision"])

print(f"Folds evaluated: {len(wf_results)}  |  Folds with ML model usable: {len(valid)}")
print(f"Mean precision — base: {valid['base_precision'].mean():.3f}  "
      f"rolling: {valid['rolling_precision'].mean():.3f}")
print(f"Mean ML training trades — rolling: {valid['train_trades_rolling'].mean():.0f}")

Folds evaluated: 11  |  Folds with ML model usable: 11
Mean precision — base: 0.495  rolling: 0.532
Mean ML training trades — rolling: 266


In [450]:
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Precision per fold", "ML training trades per fold", "Test trades: unfiltered vs. filtered",
))

x = valid["fold_id"]
for name, prec_col, color in [
    ("Base", "base_precision", "steelblue"),
    ("Rolling", "rolling_precision", "darkorange"),
]:
    fig.add_trace(go.Scatter(x=x, y=valid[prec_col], mode="lines+markers", name=name,
                              line=dict(color=color, width=2)), row=1, col=1)

fig.add_trace(go.Bar(x=x, y=valid["train_trades_rolling"], marker_color="darkorange",
                      showlegend=False), row=1, col=2)

fig.add_trace(go.Bar(x=x, y=valid["test_trades"], name="Unfiltered (base)",
                      marker_color="steelblue"), row=1, col=3)
fig.add_trace(go.Bar(x=x, y=valid["rolling_trades"], name="Filtered (rolling taken)",
                      marker_color="darkorange"), row=1, col=3)

fig.update_yaxes(range=[0, 1], title_text="Precision", row=1, col=1)
fig.update_xaxes(title_text="Fold", row=1, col=1)
fig.update_xaxes(title_text="Fold", row=1, col=2)
fig.update_xaxes(title_text="Fold", row=1, col=3)
fig.update_layout(height=700, width=1800, barmode="group", title="Walk-forward results")
fig.show()

## 10. Pooled trade-level significance tests

Fold-level comparisons have limited power — a handful of folds is a small paired sample. Pooling every
trade across all folds into one sample fixes that, at the cost of treating trades from different folds
(different parameters, different periods) as exchangeable. Two-proportion z-test: taken vs. skipped
trades — does the rolling filter's call carry real information?

In [451]:
def two_proportion_ztest(wins1, n1, wins2, n2):
    p1, p2 = wins1 / n1, wins2 / n2
    p_pool = (wins1 + wins2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se if se > 0 else np.nan
    return p1, p2, z, 2 * (1 - norm.cdf(abs(z)))


def pooled_kept_vs_skipped(df, label):
    d = df.dropna(subset=[f"{label}_trades", f"{label}_wins"])
    kept_trades, kept_wins = int(d[f"{label}_trades"].sum()), int(d[f"{label}_wins"].sum())
    total_trades, total_wins = int(d["test_trades"].sum()), int(d["test_wins"].sum())
    skipped_trades, skipped_wins = total_trades - kept_trades, total_wins - kept_wins
    return two_proportion_ztest(kept_wins, kept_trades, skipped_wins, skipped_trades) + (kept_trades, skipped_trades)


p_kept, p_skipped, z, p_value, n_kept, n_skipped = pooled_kept_vs_skipped(wf_results, "rolling")
results_summary = {"rolling": (p_kept, p_skipped, z, p_value, n_kept, n_skipped)}
print(f"Rolling: taken n={n_kept} rate={p_kept:.4f}  |  "
      f"skipped n={n_skipped} rate={p_skipped:.4f}  |  z={z:.3f}  p={p_value:.4f}")

Rolling: taken n=709 rate=0.5120  |  skipped n=756 rate=0.4630  |  z=1.876  p=0.0606


In [452]:
labels = ["Rolling\n(taken)", "Rolling\n(skipped)"]
rates = [results_summary["rolling"][0], results_summary["rolling"][1]]
ns = [results_summary["rolling"][4], results_summary["rolling"][5]]
colors = ["darkorange", "lightgray"]

ci = [wilson_ci(rate * n, n) for rate, n in zip(rates, ns)]
minus = [rate - lo for rate, (lo, hi) in zip(rates, ci)]
plus = [hi - rate for rate, (lo, hi) in zip(rates, ci)]
overall_base_rate = wf_results["test_wins"].sum() / wf_results["test_trades"].sum()

fig = go.Figure(go.Bar(x=labels, y=rates, marker_color=colors,
                        error_y=dict(type="data", array=plus, arrayminus=minus)))
fig.add_hline(y=overall_base_rate, line=dict(color="white", dash="dash", width=2),
              annotation_text="Overall base precision")
fig.update_layout(title="Pooled trade-level comparison — rolling filter", yaxis_title="Win rate (95% CI)",
                   yaxis_range=[0, 1], height=700, width=900)
fig.show()